# GDS Lab: Louvain Community Detection on Maintenance Faults

This notebook uses the Neo4j Graph Data Science (GDS) library to group aircraft into
**risk communities** based on shared fault patterns, then brings those community labels
back into Databricks to cross-reference with sensor telemetry.

## What You'll Learn
- How to build a virtual co-occurrence graph using Cypher aggregation projection
- How to run Louvain community detection on a weighted relationship graph
- How to write GDS results back to Neo4j node properties
- How to use community labels as a feature dimension in Databricks sensor analysis

## Core Idea
Two aircraft share a weighted edge in our projection if they have experienced the
same fault type — the more fault types in common, the stronger the edge. Louvain
then finds groups of aircraft that are densely connected through shared faults.
Aircraft in the same community share failure modes and may benefit from
coordinated maintenance scheduling.

## Prerequisites
- Neo4j Aura credentials (AuraDB Professional or higher — GDS plugin required)
- Full dataset loaded via `02_load_neo4j_full.ipynb`
- Workshop cluster with Neo4j Spark Connector installed

## Instructions
1. Enter your Neo4j credentials in Section 1
2. Run all cells in order

## Section 1: Configuration

In [ ]:
%pip install neo4j

In [ ]:
# ============================================
# CONFIGURATION - Enter your Neo4j credentials
# ============================================

NEO4J_URI = ""      # e.g., "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""  # Your password from Lab 1

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: Please enter your Neo4j credentials above before running!")
else:
    print(f"Configuration ready — {NEO4J_URI}")

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def run_query(cypher, params=None):
    """Execute a Cypher query and return results as a list of dicts."""
    with driver.session() as session:
        return [dict(r) for r in session.run(cypher, params or {})]

# Configure Spark Connector for bulk reads
spark.conf.set("neo4j.url", NEO4J_URI)
spark.conf.set("neo4j.authentication.basic.username", NEO4J_USERNAME)
spark.conf.set("neo4j.authentication.basic.password", NEO4J_PASSWORD)
spark.conf.set("neo4j.database", "neo4j")

def spark_query(cypher):
    """Read Neo4j query results as a Spark DataFrame."""
    return (spark.read
        .format("org.neo4j.spark.DataSource")
        .option("query", cypher)
        .load())

# Verify connectivity
result = run_query("RETURN 'Connected to Neo4j' AS status")
print(result[0]["status"])

## Section 2: Explore the Fault Landscape

Before building the community graph, understand the distribution of fault types
and how they vary across aircraft models.

In [ ]:
# Fault type distribution across the fleet
fault_dist = run_query("""
    MATCH (m:MaintenanceEvent)
    RETURN m.fault AS fault_type,
           count(*) AS occurrences,
           count(CASE WHEN m.severity = 'CRITICAL' THEN 1 END) AS critical_count
    ORDER BY occurrences DESC
""")

print(f"{'Fault Type':<30} {'Total':>8} {'Critical':>10}")
print("-" * 52)
for row in fault_dist:
    print(f"{row['fault_type']:<30} {row['occurrences']:>8} {row['critical_count']:>10}")

In [ ]:
# Which fault types dominate each aircraft model?
model_faults = run_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m:MaintenanceEvent)
    RETURN a.model AS model,
           m.fault AS fault_type,
           count(*) AS count
    ORDER BY model, count DESC
""")

current_model = None
for row in model_faults:
    if row["model"] != current_model:
        current_model = row["model"]
        print(f"\n{current_model}")
        print(f"  {'Fault':<28} {'Count':>6}")
        print(f"  {'-'*36}")
    print(f"  {row['fault_type']:<28} {row['count']:>6}")

## Section 3: Build the Fault Co-occurrence Projection

We build a virtual Aircraft-to-Aircraft graph using **Cypher aggregation projection**.
No `SHARES_FAULT` relationship exists in the database — GDS constructs it on the fly
by evaluating the Cypher query at projection time.

**Edge weight** = number of fault types the two aircraft have in common.
A pair that both suffered `bearing wear` and `overheat` gets weight 2.

> **GDS Requirement:** This cell requires GDS plugin (AuraDB Professional+).
> Run `RETURN gds.version()` in Neo4j Browser to confirm GDS is available.

In [ ]:
# Verify GDS is available
result = run_query("RETURN gds.version() AS version")
print(f"GDS version: {result[0]['version']}")

In [ ]:
# Drop the projection if it already exists
result = run_query("CALL gds.graph.drop('fault-network', false) YIELD graphName")
if result:
    print(f"Dropped existing projection: {result[0]['graphName']}")
else:
    print("No existing projection to drop.")

In [ ]:
# Build the fault co-occurrence projection.
# elementId(a1) < elementId(a2) avoids creating duplicate edges in both directions.
result = run_query("""
    MATCH (a1:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m1:MaintenanceEvent)
    MATCH (a2:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m2:MaintenanceEvent)
    WHERE elementId(a1) < elementId(a2) AND m1.fault = m2.fault
    WITH a1, a2, count(DISTINCT m1.fault) AS shared_faults
    RETURN gds.graph.project(
        'fault-network',
        a1,
        a2,
        {
            sourceNodeLabels: labels(a1),
            targetNodeLabels: labels(a2),
            relationshipType: 'SHARES_FAULT',
            relationshipProperties: {weight: shared_faults}
        },
        {undirectedRelationshipTypes: ['SHARES_FAULT']}
    )
""")

proj = result[0]
print(f"Projection created:")
print(f"  Nodes:         {proj['nodeCount']}")
print(f"  Relationships: {proj['relationshipCount']}")

## Section 4: Run Louvain Community Detection

**Louvain** is a modularity-based community detection algorithm. It iteratively
reassigns nodes to communities that maximize the density of edges within communities
relative to edges between them.

We use three execution modes:
1. `stats` — inspect community count and modularity before committing
2. `stream` — examine which aircraft land in which community
3. `write` — persist `fault_community` as a property on Aircraft nodes

In [ ]:
# Step 1: Stats — how many communities, how good is the partitioning?
result = run_query("""
    CALL gds.louvain.stats('fault-network', {relationshipWeightProperty: 'weight'})
    YIELD communityCount, modularity, ranLevels
""")

stats = result[0]
print(f"Community count:  {stats['communityCount']}")
print(f"Modularity score: {round(stats['modularity'], 4)}  (higher = better-separated communities)")
print(f"Levels run:       {stats['ranLevels']}")

In [ ]:
# Step 2: Stream — see which aircraft belong to which community
communities = run_query("""
    CALL gds.louvain.stream('fault-network', {relationshipWeightProperty: 'weight'})
    YIELD nodeId, communityId
    RETURN gds.util.asNode(nodeId).tail_number AS aircraft,
           gds.util.asNode(nodeId).model      AS model,
           gds.util.asNode(nodeId).operator   AS operator,
           communityId
    ORDER BY communityId, aircraft
""")

current_community = None
for row in communities:
    if row["communityId"] != current_community:
        current_community = row["communityId"]
        print(f"\nCommunity {current_community}")
        print(f"  {'Aircraft':<12} {'Model':<12} {'Operator'}")
        print(f"  {'-'*42}")
    print(f"  {row['aircraft']:<12} {row['model']:<12} {row['operator']}")

In [ ]:
# Step 3: Write — persist communityId as a property on each Aircraft node
result = run_query("""
    CALL gds.louvain.write('fault-network', {
        writeProperty: 'fault_community',
        relationshipWeightProperty: 'weight'
    })
    YIELD communityCount, modularity, nodePropertiesWritten
""")

r = result[0]
print(f"Written fault_community to {r['nodePropertiesWritten']} Aircraft nodes")
print(f"Communities: {r['communityCount']},  Modularity: {round(r['modularity'], 4)}")

## Section 5: Inspect the Communities

Now that each Aircraft node has a `fault_community` property, query back through
the maintenance chain to understand what defines each community.

In [ ]:
# What fault types dominate each community?
fault_profile = run_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m:MaintenanceEvent)
    WHERE a.fault_community IS NOT NULL
    RETURN a.fault_community AS community,
           m.fault            AS fault_type,
           count(*)           AS occurrences
    ORDER BY community, occurrences DESC
""")

current = None
for row in fault_profile:
    if row["community"] != current:
        current = row["community"]
        print(f"\nCommunity {current} — top faults:")
        print(f"  {'Fault Type':<28} {'Count':>6}")
        print(f"  {'-'*36}")
    print(f"  {row['fault_type']:<28} {row['occurrences']:>6}")

In [ ]:
# Aircraft models and operators per community
composition = run_query("""
    MATCH (a:Aircraft)
    WHERE a.fault_community IS NOT NULL
    RETURN a.fault_community                  AS community,
           count(a)                           AS aircraft_count,
           collect(DISTINCT a.model)          AS models,
           collect(DISTINCT a.operator)       AS operators
    ORDER BY community
""")

for row in composition:
    print(f"Community {row['community']}:")
    print(f"  Aircraft: {row['aircraft_count']}")
    print(f"  Models:   {', '.join(row['models'])}")
    print(f"  Operators:{', '.join(row['operators'])}")
    print()

## Section 6: Enrich with Sensor Data in Databricks

Pull the community assignments into Databricks and join with sensor readings.
The question: **do aircraft in the same fault community share similar sensor profiles?**

If communities have distinct sensor signatures, the community label becomes a
useful feature for predictive maintenance models.

In [ ]:
# Read Aircraft nodes with community assignments via Spark Connector
community_df = spark_query("""
    MATCH (a:Aircraft)
    WHERE a.fault_community IS NOT NULL
    RETURN a.aircraft_id    AS aircraft_id,
           a.tail_number    AS tail_number,
           a.model          AS model,
           a.operator       AS operator,
           a.fault_community AS fault_community
""")

community_df.createOrReplaceTempView("aircraft_communities")
display(community_df.orderBy("fault_community", "tail_number"))

In [ ]:
# Average sensor readings per community — do communities have distinct sensor profiles?
sensor_by_community = spark.sql("""
    SELECT
        ac.fault_community,
        sen.type                                                        AS sensor_type,
        round(AVG(r.value), 2)                                          AS avg_value,
        round(STDDEV(r.value), 3)                                       AS stddev_value,
        round(PERCENTILE(r.value, 0.95), 2)                             AS p95_value,
        count(DISTINCT ac.aircraft_id)                                  AS aircraft_in_community
    FROM sensor_readings r
    JOIN sensors sen      ON r.sensor_id  = sen.sensor_id
    JOIN systems  s       ON sen.system_id = s.system_id
    JOIN aircraft_communities ac ON s.aircraft_id = ac.aircraft_id
    GROUP BY ac.fault_community, sen.type
    ORDER BY ac.fault_community, sen.type
""")

display(sensor_by_community)

In [ ]:
# Critical event rate per community — useful for risk-ranking communities
critical_by_community = spark_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m:MaintenanceEvent)
    WHERE a.fault_community IS NOT NULL
    RETURN a.fault_community                                    AS community,
           count(m)                                             AS total_events,
           count(CASE WHEN m.severity = 'CRITICAL' THEN 1 END) AS critical_events,
           round(
               100.0 * count(CASE WHEN m.severity = 'CRITICAL' THEN 1 END) / count(m),
               1
           )                                                    AS critical_pct
    ORDER BY critical_pct DESC
""")

display(critical_by_community)

## Section 7: Clean Up

In [ ]:
# Drop the in-memory projection (the fault_community property on Aircraft nodes persists)
run_query("CALL gds.graph.drop('fault-network', false) YIELD graphName")
driver.close()
print("Projection dropped. fault_community property remains on Aircraft nodes.")

## Neo4j Browser Queries

Open [console.neo4j.io](https://console.neo4j.io) and try these queries.

**View communities:**
```cypher
MATCH (a:Aircraft)
WHERE a.fault_community IS NOT NULL
RETURN a.fault_community AS community,
       collect(a.tail_number) AS aircraft
ORDER BY community
```

**Visualize a community's fault chain:**
```cypher
MATCH (a:Aircraft {fault_community: 0})-[:HAS_SYSTEM]->(s:System)
      -[:HAS_COMPONENT]->(c:Component)-[:HAS_EVENT]->(m:MaintenanceEvent)
RETURN a, s, c, m
LIMIT 50
```

**Find the community with the most critical events:**
```cypher
MATCH (a:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
      -[:HAS_EVENT]->(m:MaintenanceEvent {severity: 'CRITICAL'})
WHERE a.fault_community IS NOT NULL
RETURN a.fault_community AS community, count(m) AS critical_events
ORDER BY critical_events DESC
```